In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sqlite3
import glob

import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parent))

from utils import *
from utils_figures import *

In [ ]:
"""
Computes ALL metrics for a single run in one go to minimize I/O.
"""

base_dir = 'experiments_recsys'
network = 'BA'
recsys = 'RC'
run = 0

#### Network information #######

with sqlite3.connect(db_path) as conn:
    run_folder = os.path.dirname(get_db_path(base_dir, network, recsys, run))
    csv_files = glob.glob(os.path.join(run_folder, "*.csv"))
    author_degrees = {}

    df_edges = pd.read_csv(csv_files[0], header=None, names=['source', 'target'])
    deg_counts = pd.concat([df_edges['source'], df_edges['target']]).value_counts()
    deg_dict = deg_counts.to_dict()
    degrees = {k: int(v/2) for k,v in deg_dict.items()}
    
    query_user = "SELECT username, id FROM user_mgmt"
    df_users = pd.read_sql(query_user, conn)
    
    username_to_id = dict(zip(df_users["username"], df_users["id"]))
    id_to_username = dict(zip(df_users["id"], df_users["username"]))
    
    degrees_by_id = {
        username_to_id[user]: deg
        for user, deg in degrees.items()
        if user in username_to_id
        }
    
    return degrees_by_id

    

################################

db_path = get_db_path(base_dir, network, recsys, run)
print(db_path)
if not os.path.exists(db_path):
    print(f"Warning: DB not found at {db_path}")
    exit
         
with sqlite3.connect(db_path) as conn:
    conn.row_factory = sqlite3.Row
    cursor = conn.cursor()

    query_posts = "SELECT round AS recommended_at, post_ids, user_id FROM recommendations"
    df_recommendations = pd.read_sql(query_posts, conn)
    
    df_recommendations['pids_list'] = df_recommendations['post_ids'].apply(lambda pids_str: pids_to_list(pids_str))
    df_recommendations.drop(columns=['post_ids'], inplace=True)
    df_recommendations = df_recommendations.explode('pids_list')
    df_recommendations.rename(columns={"pids_list": "post_id"}, inplace=True)
    
    df_recommendations["viewer_username"] = df_recommendations["user_id"].map(id_to_username)

    df_recommendations["degree"] = df_recommendations["viewer_username"].map(degrees)  
    
    df_recommendations = df_recommendations.rename(columns={
    "degree": "viewer_degree",
    "user_id": "viewer_id"  
    })
        
    query_posts = "SELECT id, round AS created_at, user_id FROM post" 
    df_posts = pd.read_sql(query_posts, conn) 
    df_posts = df_posts.rename(columns={ "id": "post_id", "user_id": "author_id" })
    df_recommendations = df_recommendations.merge(
        df_posts,
        on="post_id",
        how="left"
    )
    df_recommendations['author_username'] = df_recommendations["author_id"].map(id_to_username)
    df_recommendations["author_degree"] = df_recommendations["author_username"].map(degrees)  
    
    query = "SELECT post_id, user_id, type, round AS reacted_at FROM reactions"
    df_reactions = pd.read_sql(query, conn)  
    
    df_recommendations = df_recommendations.merge(
    df_reactions[["post_id", "user_id", "type", "reacted_at"]],
    how="left",
    left_on=["recommended_at", "post_id", "viewer_id"],
    right_on=["reacted_at", "post_id", "user_id"])
    
    df_recommendations['reacted_at'] = df_recommendations['reacted_at'].fillna(-1).astype(int)
    df_recommendations['viewer_reacted'] = df_recommendations['type'].notna().astype(int)
    df_recommendations['type'] = df_recommendations['type'].fillna('None')
    df_recommendations['viewer_degree'] = df_recommendations['viewer_id'].map(degrees_by_id)
    df_recommendations = df_recommendations.drop(
    columns=['user_id', 'viewer_username', 'author_username'],
    errors='ignore'  # won't crash if any column is missing
    )
    
post_reactions = df_recommendations.groupby(['author_id', 'post_id', 'recommended_at', 'reacted_at', 'created_at']).agg({'viewer_reacted': 'sum', 'viewer_id': 'count', 'viewer_degree': list}).reset_index()
post_reactions.rename(columns={'viewer_reacted': 'num_reactions', 'viewer_id': 'num_viewers'}, inplace=True)
post_reactions['reaction_rate'] = post_reactions['num_reactions'] / post_reactions['num_viewers']
post_reactions['age_at_recommendation'] = post_reactions['recommended_at'] - post_reactions['created_at']

post_reactions["author_degree"] = post_reactions["author_id"].map(degrees_by_id)

../experiments_recsys/BA_RC_0/database_server.db


In [13]:
post_reactions

,author_id,post_id,recommended_at,reacted_at,created_at,num_reactions,num_viewers,viewer_degree,reaction_rate,age_at_recommendation,author_degree
0,2,8044,128,-1,128,0,9,"[9, 6, 6, 6, 6, 6, 6, 6, 6]",0.0,0,78
1,2,25116,399,-1,398,0,12,"[20, 20, 5, 5, 5, 5, 7, 7, 16, 16, 16, 16]",0.0,1,78
2,2,42294,667,-1,667,0,7,"[5, 5, 7, 5, 5, 5, 5]",0.0,0,78
3,2,42294,667,667,667,2,2,"[6, 6]",1.0,0,78
4,2,42295,667,-1,667,0,5,"[5, 5, 7, 6, 6]",0.0,0,78
...,...,...,...,...,...,...,...,...,...,...,...
149414,1001,90355,1429,1429,1429,3,3,"[10, 10, 10]",1.0,0,5
149415,1001,90355,1430,-1,1429,0,4,"[8, 10, 10, 10]",0.0,1,5
149416,1001,90356,1429,-1,1429,0,5,"[10, 10, 10, 6, 6]",0.0,0,5
149417,1001,90356,1430,-1,1429,0,1,[8],0.0,1,5


In [14]:
post_reactions.describe()

,author_id,post_id,recommended_at,reacted_at,created_at,num_reactions,num_viewers,reaction_rate,age_at_recommendation,author_degree
count,149419.000000,149419.000000,149419.000000,149419.000000,149419.000000,149419.000000,149419.000000,149419.000000,149419.000000,149419.000000
mean,504.005608,45615.277381,720.490687,261.446342,720.362323,1.899558,5.628454,0.363548,0.128364,10.036200
std,295.620853,26295.954427,415.873831,428.233990,415.874605,3.322434,3.251367,0.481022,0.334496,10.240177
min,2.000000,1.000000,1.000000,-1.000000,1.000000,0.000000,1.000000,0.000000,0.000000,5.000000
25%,234.000000,22872.500000,361.000000,-1.000000,361.000000,0.000000,3.000000,0.000000,0.000000,5.000000
50%,510.000000,45612.000000,719.000000,-1.000000,719.000000,0.000000,5.000000,0.000000,0.000000,7.000000
75%,761.000000,68356.500000,1079.000000,451.000000,1079.000000,3.000000,8.000000,1.000000,0.000000,10.000000
max,1001.000000,91230.000000,1440.000000,1440.000000,1440.000000,37.000000,37.000000,1.000000,1.000000,141.000000


In [32]:
print(post_reactions)

        author_id  post_id  recommended_at  reacted_at  created_at  \
0               2     8044             128          -1         128   
1               2    25116             399          -1         398   
2               2    42294             667          -1         667   
3               2    42294             667         667         667   
4               2    42295             667          -1         667   
...           ...      ...             ...         ...         ...   
149414       1001    90355            1429        1429        1429   
149415       1001    90355            1430          -1        1429   
149416       1001    90356            1429          -1        1429   
149417       1001    90356            1430          -1        1429   
149418       1001    90356            1430        1430        1429   

        num_reactions  num_viewers  \
0                   0            9   
1                   0           12   
2                   0            7   
3      

In [43]:
post_reactions

,author_id,post_id,recommended_at,reacted_at,created_at,num_reactions,num_viewers,viewer_degree,reaction_rate,age_at_recommendation,author_degree
0,2,8044,128,-1,128,0,9,"[9, 6, 6, 6, 6, 6, 6, 6, 6]",0.0,0,78
1,2,25116,399,-1,398,0,12,"[20, 20, 5, 5, 5, 5, 7, 7, 16, 16, 16, 16]",0.0,1,78
2,2,42294,667,-1,667,0,7,"[5, 5, 7, 5, 5, 5, 5]",0.0,0,78
3,2,42294,667,667,667,2,2,"[6, 6]",1.0,0,78
4,2,42295,667,-1,667,0,5,"[5, 5, 7, 6, 6]",0.0,0,78
...,...,...,...,...,...,...,...,...,...,...,...
149414,1001,90355,1429,1429,1429,3,3,"[10, 10, 10]",1.0,0,5
149415,1001,90355,1430,-1,1429,0,4,"[8, 10, 10, 10]",0.0,1,5
149416,1001,90356,1429,-1,1429,0,5,"[10, 10, 10, 6, 6]",0.0,0,5
149417,1001,90356,1430,-1,1429,0,1,[8],0.0,1,5


In [ ]:
import numpy as np
import pandas as pd

def compute_recs_vs_degree(df, degree_bins):
    """
    Compute average total recommendations per author
    as a function of author degree (binned).
    
    Parameters
    ----------
    df : pandas.DataFrame
        Single run dataframe (post_reaction) with columns:
        - author_id
        - author_degree
    degree_bins : array-like
        Fixed bin edges (must be same across runs)
        
    Returns
    -------
    bin_centers : np.array
    bin_means : np.array
    """
    
    # --- aggregate at author level ---
    author_recs = (
        df.groupby('author_id')
          .size()
          .reset_index(name='total_recommendations')
    ) #number of times a post of user u was recommended across all rounds (i.e., sum of recommendations of all posts of user u)
    
    author_degree = (
        df.groupby('author_id')['author_degree']
          .first()
          .reset_index()
    ) #author degree (same for all posts of the same author)
    
    author_stats = author_recs.merge(author_degree, on='author_id') #merge to have both total recommendations and degree for each author
    
    # --- bin by degree ---
    author_stats['degree_bin'] = pd.cut(
        author_stats['author_degree'],
        bins=degree_bins,
        include_lowest=True
    ) #
    
    # --- compute mean per bin ---
    bin_means = (
        author_stats
        .groupby('degree_bin')['total_recommendations']
        .mean()
    )
    
    # Ensure all bins are present (important for averaging)
    bin_means = bin_means.reindex(
        pd.IntervalIndex.from_breaks(degree_bins),
        fill_value=np.nan
    )
    
    # Bin centers (for plotting)
    bin_centers = 0.5 * (degree_bins[:-1] + degree_bins[1:])
    
    return bin_centers, bin_means.values

In [ ]:
from collections import defaultdict
import numpy as np
import pandas as pd
from tqdm import tqdm

def agg_runs_recs_vs_degree():
    metrics = defaultdict(dict)
    
    total_steps = len(NETWORKS) * len(RECSYSS) * len(RUNS)
    pbar = tqdm(total=total_steps, desc='Processing runs')

    for network in NETWORKS:
        for recsys in RECSYSS:
            
            all_runs = []
            
            # --- Collect per-run author-level data ---
            for run in RUNS:
                author_stats = run_analysis(network, recsys, run)
                pbar.update(1)
                
                if author_stats is not None:
                    all_runs.append(author_stats)
            
            if not all_runs:
                continue
            
            # --------------------------------------------------
            # 1️⃣ Define fixed degree bins across runs
            # --------------------------------------------------
            
            max_degree = max(df['author_degree'].max() for df in all_runs)
            
            bins = np.logspace(
                np.log10(1),
                np.log10(max_degree),
                num=20
            )
            
            # --------------------------------------------------
            # 2️⃣ Compute per-run binned averages
            # --------------------------------------------------
            
            aligned_runs = []
            
            for df in all_runs:
                
                df = df.copy()
                
                df['degree_bin'] = pd.cut(
                    df['author_degree'],
                    bins=bins,
                    include_lowest=True
                )
                
                bin_means = (
                    df.groupby('degree_bin')['total_recommendations']
                      .mean()
                )
                
                # Align bins explicitly
                bin_means = bin_means.reindex(
                    pd.IntervalIndex.from_breaks(bins),
                    fill_value=np.nan
                )
                
                aligned_runs.append(bin_means.values)
            
            # --------------------------------------------------
            # 3️⃣ Aggregate across runs
            # --------------------------------------------------
            
            df_aligned = pd.DataFrame(aligned_runs)
            
            mean_curve = df_aligned.mean(axis=0)
            std_curve = df_aligned.std(axis=0)
            
            metrics[network][recsys] = {
                'mean': mean_curve,
                'std': std_curve,
                'bins': bins
            }

    pbar.close()
    return metrics

In [ ]:
import sys
import os
import glob
import sqlite3
import gc
import pickle
import numpy as np
import pandas as pd
import matplotlib
from matplotlib import rcParams
import matplotlib.pyplot as plt
from cycler import cycler
from collections import defaultdict, Counter
from tqdm import tqdm
import networkx as nx

from utils import *
from utils_figures import *

matplotlib.use('Agg') 


def run_analysis(base_dir, network, recsys, run):
    """
    Returns a dataframe with:
        author_id
        total_recommendations
        author_degree
        unique_reach   <-- number of distinct users who were recommended the author's posts
    for a single run.
    """
    
    #### Network information #######
    degrees = compute_author_degrees(base_dir, network, recsys, run, by='id')
    ################################
    
    db_path = get_db_path(base_dir, network, recsys, run)

    if not os.path.exists(db_path):
        print(f"DB not found: {db_path}")
        return None

    # --------------------------------------------
    # Load recommendations
    # --------------------------------------------
    with sqlite3.connect(db_path) as conn:

        df_rec = pd.read_sql(
            "SELECT round AS recommended_at, post_ids, user_id FROM recommendations",
            conn
        )

        df_rec['post_id'] = df_rec['post_ids'].apply(pids_to_list)
        df_rec = df_rec.explode('post_id')
        df_rec = df_rec.rename(columns={'user_id': 'viewer_id'})
        df_rec.drop(columns=['post_ids'], inplace=True)

        df_posts = pd.read_sql(
            "SELECT id AS post_id, user_id AS author_id FROM post",
            conn
        )

        df_rec = df_rec.merge(df_posts, on="post_id", how="left")

    # --------------------------------------------
    # Map author degree
    # --------------------------------------------
    df_rec['author_degree'] = df_rec['author_id'].map(degrees)

    # --------------------------------------------
    # Aggregate at author level
    # --------------------------------------------
    author_stats = (
        df_rec.groupby('author_id')
        .agg(
            total_recommendations=('post_id', 'count'),
            author_degree=('author_degree', 'first'),
            unique_reach=('viewer_id', 'nunique')
        )
        .reset_index()
    )

    # Compute delta_recs
    author_stats['delta_recs'] = compute_delta_recs(df_rec).reindex(author_stats['author_id']).values
    
    print(f"Run {network}-{recsys}-{run}: {len(author_stats)} authors with recommendations")
    print(author_stats.sort_values('author_degree'))
    return author_stats

# ==================================================
# 2️⃣ AGGREGATE OVER RUNS
# ==================================================

def aggregate_runs(base_dir, networks, recsyss, runs, n_bins=20):
    """
    Aggregates runs and computes binned mean/STD for:
        - total_recommendations
        - unique_reach
    Returns a nested dict:
        metrics[network][recsys] = {
            'total_recommendations': {'mean': ..., 'std': ..., 'bins': ...},
            'unique_reach': {'mean': ..., 'std': ..., 'bins': ...}
        }
    """
    
    metrics = defaultdict(dict)
    total_steps = len(networks) * len(recsyss) * len(runs)
    pbar = tqdm(total=total_steps, desc="Processing runs")

    for network in networks:
        for recsys in recsyss:

            all_runs = []

            for run in runs:
                print('Processing:', network, recsys, run)
                df = run_analysis(base_dir, network, recsys, run)
                pbar.update(1)
                if df is not None:
                    all_runs.append(df)

            if not all_runs:
                continue

            # ----------------------------------------
            # Define fixed log bins across runs
            # ----------------------------------------
            all_degrees = np.concatenate([df['author_degree'].dropna().values for df in all_runs])
            bins = np.logspace(0, np.log10(max(all_degrees)), num=n_bins, dtype=int)
            bins = np.unique(bins)

            # ----------------------------------------
            # Compute per-run binned means for both metrics
            # ----------------------------------------
            # ----------------------------------------
            # Compute per-run binned means for all metrics
            # ----------------------------------------
            aligned_tot = []
            aligned_reach = []
            aligned_delta = []

            for df in all_runs:
                df = df.copy()
                df['degree_bin'] = pd.cut(df['author_degree'], bins=bins, include_lowest=True)

                # Average total recommendations per bin
                bin_means_tot = df.groupby('degree_bin')['total_recommendations'].mean()
                bin_means_tot = bin_means_tot.reindex(pd.IntervalIndex.from_breaks(bins), fill_value=np.nan)
                aligned_tot.append(bin_means_tot.values)

                # Average unique reach per bin
                bin_means_reach = df.groupby('degree_bin')['unique_reach'].mean()
                bin_means_reach = bin_means_reach.reindex(pd.IntervalIndex.from_breaks(bins), fill_value=np.nan)
                aligned_reach.append(bin_means_reach.values)

                # Average delta_recs per bin
                bin_means_delta = df.groupby('degree_bin')['delta_recs'].mean()
                bin_means_delta = bin_means_delta.reindex(pd.IntervalIndex.from_breaks(bins), fill_value=np.nan)
                aligned_delta.append(bin_means_delta.values)

            # Convert to DataFrame for aggregation
            df_tot = pd.DataFrame(aligned_tot)
            df_reach = pd.DataFrame(aligned_reach)
            df_delta = pd.DataFrame(aligned_delta)

            metrics[network][recsys] = {
                'total_recommendations': {
                    'mean': df_tot.mean(axis=0),
                    'std': df_tot.std(axis=0),
                    'bins': bins
                },
                'unique_reach': {
                    'mean': df_reach.mean(axis=0),
                    'std': df_reach.std(axis=0),
                    'bins': bins
                },
                'delta_recs': {
                    'mean': df_delta.mean(axis=0),
                    'std': df_delta.std(axis=0),
                    'bins': bins
                }
            }

            print(f"Aggregated {len(all_runs)} runs for {network}-{recsys}")
            print(metrics[network][recsys])

    pbar.close()
    return metrics
    
# =============================================================================
# MAIN EXECUTION
# =============================================================================

if __name__ == "__main__":

    old_stdout = sys.stdout
    log_file = open("res/recs_vs_degree.log", "w")
    sys.stdout = log_file

    print("Starting Analysis Pipeline...")

    # -------------------------------
    # Aggregate metrics across runs
    # -------------------------------
    metrics = aggregate_runs(BASE_DIR, NETWORKS, RECSYSS, RUNS, n_bins=20)

    # -------------------------------
    # Plot total recommendations
    # ------------------------------
    
    plot_recs_vs_degree(
        metrics,
        networks=['ER','BA'],
        save_path='figs/recommendations_vs_degree/',
        scale='log',
        metrics_to_plot=['total_recommendations', 'delta_recs', 'unique_reach']
    )

    print("Ending Analysis Pipeline...")

    sys.stdout = old_stdout
    log_file.close()